# Improve DeepCAD

In this notebook the idea to improve DeepCAD by splitting up the CAD-sequences in smaller units will be explored.

__IMPORTANT__

- For this I disabled the sampling of point clouds to 2048 points, here I use the whole 8096 points

In [3]:
import os
import shutil
import h5py
import numpy as np
import sys
import pandas as pd
import json
import matplotlib.pyplot as plt
%matplotlib inline

sys.path.append("..")
sys.path.append("../code")

from dataset import PointCloudEmbeddingSequenceDataset
from models.DeepCAD.cadlib.visualize import vec2CADsolid
from OCC.Core.BRepCheck import BRepCheck_Analyzer
from OCC.Extend.DataExchange import write_step_file
from OCC.Core.STEPControl import STEPControl_Reader
from OCC.Core.StlAPI import StlAPI_Writer
from OCC.Core.BRepMesh import BRepMesh_IncrementalMesh
from models.DeepCAD.cadlib.extrude import CADSequence
from models.DeepCAD.cadlib.visualize import create_CAD
from models.DeepCAD.cadlib.visualize import CADsolid2pc
from models.DeepCAD.utils.pc_utils import write_ply

In [4]:
def get_data(dataset, index):
    data = dataset[index]
    point_cloud = data['pc']
    sequence = np.asarray(data['tgt_vec'])
    return point_cloud, sequence

In [5]:
def copy_to_temp(dataset, idx):
    cad_seq_path = dataset.get_cad_seq_path(idx)
    pc_path = dataset.get_pc_path(idx)
    print(idx)
    print(pc_path)
    json_path = pc_path.replace("pc_cad", "cad_json")
    json_path = json_path.replace("ply", "json")
    print(pc_path, json_path)
    
    temp_dir = "../data/temporary"
    if os.path.exists(temp_dir):
        shutil.rmtree(temp_dir)
    os.makedirs(temp_dir)

    dest_path_cad_seq = os.path.join(temp_dir, os.path.basename(cad_seq_path))
    dest_path_pc = os.path.join(temp_dir, os.path.basename(pc_path))
    dest_path_json = os.path.join(temp_dir, os.path.basename(json_path))

    shutil.copy(cad_seq_path, dest_path_cad_seq)
    shutil.copy(pc_path, dest_path_pc)
    shutil.copy(json_path, dest_path_json)
    
    print(f"Copied {cad_seq_path} to {dest_path_cad_seq}")
    print(f"Copied {pc_path} to {dest_path_pc}")
    print(f"Copied {json_path} to {dest_path_json}")
    return dest_path_json

def change_keys(h5_file):
    """Changes the keys from 'vec' to 'out_vec' in order to be able to show the sample using show.py"""
    with h5py.File(h5_file, 'r+') as hf:

        if 'vec' in hf:
            data = hf['vec'][:]
            hf.create_dataset('out_vec', data=data)
            del hf['vec']
            print(f"Changed keys from 'vec' to 'out_vec' in {h5_file}")

def export2step(json_path):
    filter = True
    save_path = os.path.join(*json_path.split("/")[:-1], os.path.splitext(os.path.basename(json_path))[0] + '.step')

    with open(json_path, "r") as fp:
        data = json.load(fp)

    cad_seq = CADSequence.from_dict(data)
    cad_seq.normalize()
    shape = create_CAD(cad_seq)

    write_step_file(shape, save_path)
    return save_path

def step2stl(step_path):

    save_path = os.path.join(*step_path.split("/")[:-1], os.path.splitext(os.path.basename(step_path))[0] + '.stl')
    step_reader = STEPControl_Reader()
    step_reader.ReadFile(step_path)
    step_reader.TransferRoots()
    shape = step_reader.OneShape()

    BRepMesh_IncrementalMesh(shape, 0.1)

    stl_writer = StlAPI_Writer()
    stl_writer.Write(shape, save_path)
    print(f"Wrote stl file to {save_path}")

def visualize_gt(dataset, idx):
    dest_path_h5 = copy_to_temp(dataset, idx)
   # change_keys(dest_path_h5)
    step_path = export2step(dest_path_h5)
    step2stl(step_path)

In [6]:
dataset = PointCloudEmbeddingSequenceDataset("../data", 'train', use_normals=False)

In [7]:
seq = [
    [4, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
    [0, 223, 128, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
    [0, 223, 223, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
    [0, 128, 223, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
    [0, 128, 128, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
    [5, -1, -1, -1, -1, -1, 128, 128, 128, 128, 128, 0, 128, 256, 128, 0, 0],
    [4, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
    [2, 128, 128, -1, -1, 95, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
    [5, -1, -1, -1, -1, -1, 128, 128, 128, 192, 192, 128, 64, 192, 128, 0, 0]
]
eos_row = [3, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1]

In [8]:
seq_np = np.array(seq, dtype=np.float32)
num_pad_rows = 60 - seq_np.shape[0]
pad_array = np.tile(eos_row, (num_pad_rows, 1))
seq_np_pad = np.concatenate([seq_np, pad_array], axis=0)  
seq_len = seq_np_pad[:,0].tolist().index(3)
cad_seq = CADSequence.from_vector(seq_np_pad, is_numerical=True)
custom_shape = create_CAD(cad_seq)
write_step_file(custom_shape, "a.step")
step2stl("a.step")


*******************************************************************
******        Statistics on Transfer (Write)                 ******

*******************************************************************
******        Transfer Mode = 0  I.E.  As Is       ******
******        Transferring Shape, ShapeType = 0                      ******
** WorkSession : Sending all data
 Step File Name : a.step(655 ents)  Write  Done
Wrote stl file to a.stl


In [9]:
out_pc = CADsolid2pc(custom_shape, 8096, "aha")
write_ply(out_pc, "aha.ply")

In [10]:
seq_np_pad.shape

(60, 17)

In [11]:
visualize_gt(dataset, i)

58
../data/pc_cad/0051/00516650.ply
../data/pc_cad/0051/00516650.ply ../data/cad_json/0051/00516650.json
Copied ../data/cad_vec/0051/00516650.h5 to ../data/temporary/00516650.h5
Copied ../data/pc_cad/0051/00516650.ply to ../data/temporary/00516650.ply
Copied ../data/cad_json/0051/00516650.json to ../data/temporary/00516650.json

*******************************************************************
******        Statistics on Transfer (Write)                 ******

*******************************************************************
******        Transfer Mode = 0  I.E.  As Is       ******
******        Transferring Shape, ShapeType = 0                      ******
** WorkSession : Sending all data
 Step File Name : ../data/temporary/00516650.step(1740 ents)  Write  Done
Wrote stl file to ../data/temporary/00516650.stl


## START

We will use the below sequence to create a minimum working example.

In [12]:
seq = [
    [4, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
    [0, 223, 128, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
    [0, 223, 223, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
    [0, 128, 223, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
    [0, 128, 128, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
    [5, -1, -1, -1, -1, -1, 128, 128, 128, 128, 128, 0, 128, 256, 128, 0, 0], # This creates a 1x1x1 cube at the bottom-back-right position of the unit cube
    [4, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
    [2, 128, 128, -1, -1, 95, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
    [5, -1, -1, -1, -1, -1, 128, 128, 128, 192, 192, 128, 64, 192, 128, 0, 0] # This adds a 0.5 high cylinder of radius 0.5 on top of the cube
]

#### 1. Split CAD-sequence into extrusions

In [13]:
def seq2shape(seq):
    cad_seq = CADSequence.from_vector(seq, is_numerical=True)
    shape = create_CAD(cad_seq)
    return shape

In [14]:
def shape2cad(shape, name=None):
    if name is None:
        name = "test"
    write_step_file(shape, os.path.join("examples", name + ".step"))
    step2stl(os.path.join("examples",name + ".step"))

In [15]:
def seq2CAD(seq, name=None):
    """Takes (60,17) sequence and turns it to stl file."""
    shape = seq2shape(seq)
    shape2cad(shape, name=name)

In [16]:
def pad_seq(seq):
    """Takes custom sequence (N,17) and pads it to (60,17)"""
    eos_row = [3] + 16 * [-1]
    seq_np = np.array(seq, dtype=np.float32)
    num_pad_rows = 60 - seq_np.shape[0]
    pad_array = np.tile(eos_row, (num_pad_rows, 1))
    seq_np_pad = np.concatenate([seq_np, pad_array], axis=0)  
    return seq_np_pad
    

In [17]:
def split_and_pad_sequence_by_extrusion(matrix, delimiter=5):
    """Takes (60,17) sequence and splits it by the extrusions and pads it and returns a (60,17) for each extrusion""" 
    matrix = np.array(matrix)
    assert matrix.shape == (60, 17), "Input must be (60, 17)"

    commands = matrix[:,0]
    split_indices = []
    start_idx = 0

    # Find split points
    for idx, val in enumerate(commands):
        if val == delimiter:
            split_indices.append((start_idx, idx))
            start_idx = idx + 1

    # Split and pad
    output = []
    for start, end in split_indices:
        length = 59 - (end - start)
        pad_row = [[3] + 16 * [-1]] * length
        new_matrix = matrix[start:end+1]
       
        pad_matrix = np.vstack([new_matrix, pad_row])
        output.append(pad_matrix)
    return output

In [18]:
def seq2pc(seq, nr_points=8096, name=None):
    shape = seq2shape(seq)
    if name is None:
        name = "test"
    out_pc = CADsolid2pc(shape, nr_points, name)
    write_ply(out_pc, os.path.join("examples",name + ".ply"))
    return out_pc

In [19]:
seq = [
    [4, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
    [0, 223, 128, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
    [0, 223, 223, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
    [0, 128, 223, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
    [0, 128, 128, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
    [5, -1, -1, -1, -1, -1, 128, 128, 128, 128, 128, 0, 128, 256, 128, 0, 0], # This creates a 1x1x1 cube at the bottom-back-right position of the unit cube
    [4, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
    [0, 223, 128, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
    [0, 223, 223, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
    [0, 128, 223, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
    [0, 128, 128, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
    [5, -1, -1, -1, -1, -1, 128, 128, 128, 64, 128, 0, 128, 256, 128, 1, 0], # This creates a 1x1x1 cube at the bottom-back-right position of the unit cube
    [4, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
    [2, 128, 128, -1, -1, 95, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
    [5, -1, -1, -1, -1, -1, 128, 128, 128, 192, 192, 128, 0, 192, 128, 0, 0] # This adds a 0.5 high cylinder of radius 0.5 on top of the cube
]

In [20]:
seq = [
    [4, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
    [0, 223, 128, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
    [0, 223, 223, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
    [0, 128, 223, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
    [0, 128, 128, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
    [5, -1, -1, -1, -1, -1, 128, 128, 128, 128, 128, 0, 128, 256, 128, 0, 0], # This creates a 1x1x1 cube at the bottom-back-right position of the unit cube
    [4, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
    [2, 128, 128, -1, -1, 95, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
    [5, -1, -1, -1, -1, -1, 128, 128, 128, 192, 192, 64, 64, 192, 128, 3, 0] # This adds a 0.5 high cylinder of radius 0.5 on top of the cube
]

In [21]:
sequence = pad_seq(seq)

In [22]:
from IPython.display import display, JSON
def show_json(path):
    json_path = path
    print(json_path)
    with open(json_path, "r") as file:
        data = json.load(file)
    display(JSON(data))
    return data

In [23]:
dataset = PointCloudEmbeddingSequenceDataset("../data", 'train', use_normals=False)


In [24]:
import h5py

with h5py.File("../data/cad_vec/0027/00271614.h5", 'r') as f: ##00270073
    a = f['vec'][:]  # Read the data into memory (as a numpy array)

print(a)  # Now 'a' is a numpy array and accessible


[[  4  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1]
 [  0 223 128  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1]
 [  0 223 191  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1]
 [  0 176 223  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1]
 [  0 128 191  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1]
 [  0 128 128  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1]
 [  4  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1]
 [  0 206 129  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1]
 [  0 206 192  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1]
 [  0 145 192  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1]
 [  0 145 129  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1]
 [  5  -1  -1  -1  -1  -1 192  64 192 113 128 116  68 224 128   0   0]
 [  4  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1]
 [  0 223 128  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1]
 [  0 

In [25]:
index = 13
data = dataset[index]
id = dataset.get_id(index)
print(id)
print(dataset.get_cad_seq_path(index))
sequence = data["tgt_vec"].numpy()
sequence, flag = check_sequence(sequence, id)
print(sequence.shape)
for i, command in enumerate(sequence):
    if command[0] == 3:
        print(i)
        break
       
    print(i, end="")
    print(command)

00271614
../data/cad_vec/0027/00271614.h5


NameError: name 'check_sequence' is not defined

In [26]:
sequence

array([[  4,  -1,  -1, ...,  -1,  -1,  -1],
       [  0, 223, 128, ...,  -1,  -1,  -1],
       [  0, 223, 191, ...,  -1,  -1,  -1],
       ...,
       [  3,  -1,  -1, ...,  -1,  -1,  -1],
       [  3,  -1,  -1, ...,  -1,  -1,  -1],
       [  3,  -1,  -1, ...,  -1,  -1,  -1]])

In [27]:
json_path = os.path.join("..", "data", "cad_json", id[:4], id + ".json")
json_data = show_json(json_path)

../data/cad_json/0027/00271614.json


<IPython.core.display.JSON object>

In [28]:
def get_extrusion_distance_per_extrusion(data):
    distances_per_extrusion = []
    for item in json_data["sequence"]:
        if item["type"] == "ExtrudeFeature":
            
            extrude_id = item["entity"]
            extrude_entity = data["entities"][extrude_id]
            num_profiles = len(extrude_entity["profiles"])
            
            extent_one = extrude_entity["extent_one"]["distance"]["value"]
            extent_two = extrude_entity["extent_two"]["distance"]["value"]
            
            print("Extrude Type:", extrude_entity["extent_type"])
            print("Extent One:", extent_one)
            print("Extent Two:", extent_two)
            print("Number of Profiles:", num_profiles)

            extrusion_distances = (extent_one, extent_two)

            for i in range(num_profiles):
                distances_per_extrusion.append(extrusion_distances)
            print()
    return distances_per_extrusion
    

In [29]:
distances = get_extrusion_distance_per_extrusion(json_data)

Extrude Type: OneSideFeatureExtentType
Extent One: 5.1816
Extent Two: 0.0
Number of Profiles: 1

Extrude Type: OneSideFeatureExtentType
Extent One: -0.025400000000000002
Extent Two: 0.0
Number of Profiles: 1

Extrude Type: OneSideFeatureExtentType
Extent One: -0.025400000000000002
Extent Two: 0.0
Number of Profiles: 2



In [30]:
distances

[(5.1816, 0.0),
 (-0.025400000000000002, 0.0),
 (-0.025400000000000002, 0.0),
 (-0.025400000000000002, 0.0)]

In [31]:
shape = seq2CAD(sequence, "round")
seq2pc(sequence, name="round")


*******************************************************************
******        Statistics on Transfer (Write)                 ******

*******************************************************************
******        Transfer Mode = 0  I.E.  As Is       ******
******        Transferring Shape, ShapeType = 0                      ******
Wrote stl file to examples/round.stl
** WorkSession : Sending all data
 Step File Name : examples/round.step(1722 ents)  Write  Done


TrackedArray([[ 0.3189967 , -0.23434317, -0.03887571],
              [-0.02615326, -0.05163523, -0.09375   ],
              [-0.02212171, -0.15608752, -0.06176315],
              ...,
              [-0.1171875 , -0.55912242,  0.15141721],
              [ 0.33891032, -0.75      , -0.02401332],
              [-0.08063981, -0.51123446, -0.09375   ]])

In [32]:
a = [

[  4,  -1,  -1,  -1,  -1,  -1,  -1,  -1,  -1,  -1,  -1,  -1,  -1,  -1,  -1,  -1,  -1],
[  1, 128, 128,  255,   1,  -1,  -1,  -1,  -1,  -1,  -1,  -1,  -1,  -1,  -1,  -1,  -1],
[  0, 128, 129,  -1,  -1,  -1,  -1,  -1,  -1,  -1,  -1,  -1,  -1,  -1,  -1,  -1,  -1],
[  0, 128, 127,  -1,  -1,  -1,  -1,  -1,  -1,  -1,  -1,  -1,  -1,  -1,  -1,  -1,  -1],
[  4,  -1,  -1,  -1,  -1,  -1,  -1,  -1,  -1,  -1,  -1,  -1,  -1,  -1,  -1,  -1,  -1],
[  2, 175, 128,  -1,  -1,  34,  -1,  -1,  -1,  -1,  -1,  -1,  -1,  -1,  -1,  -1,  -1],
[  5,  -1,  -1,  -1,  -1,  -1, 128, 128, 128, 128, 128, 128,  96, 165, 128,   0,   0]
]

b = pad_seq(a)

In [33]:
extrusion_splits_seq = split_and_pad_sequence_by_extrusion(b)

In [34]:
extrusion_splits_seq.shape

AttributeError: 'list' object has no attribute 'shape'

In [35]:
for i, ext_seq in enumerate(extrusion_splits_seq):
    seq2CAD(ext_seq, name=str(i) + "_test")
    seq2pc(ext_seq, name=str(i) + "_test")

Wrote stl file to examples/0_test.stl
*******************************************************************
******        Statistics on Transfer (Write)                 ******


*******************************************************************
******        Transfer Mode = 0  I.E.  As Is       ******
******        Transferring Shape, ShapeType = 2                      ******
** WorkSession : Sending all data
 Step File Name : examples/0_test.step(388 ents)  Write  Done


In [36]:
seq2CAD(b, "final")
seq2pc(b, name="final")

Wrote stl file to examples/final.stl
*******************************************************************
******        Statistics on Transfer (Write)                 ******


*******************************************************************
******        Transfer Mode = 0  I.E.  As Is       ******
******        Transferring Shape, ShapeType = 2                      ******
** WorkSession : Sending all data
 Step File Name : examples/final.step(388 ents)  Write  Done


TrackedArray([[ 0.56455649, -0.21479177,  0.09037158],
              [ 0.04507802, -0.16820825,  0.02248979],
              [ 0.48538506,  0.27291914,  0.10181122],
              ...,
              [ 0.14810908, -0.25809244,  0.2890625 ],
              [ 0.15228908, -0.27740102,  0.090328  ],
              [ 0.02873785,  0.12894692,  0.19819022]])

In [37]:
extrusion_splits_seq = split_and_pad_sequence_by_extrusion(sequence)

In [38]:
for ext in extrusion_splits_seq:
    for command in ext:
        if command[0] == 3:
            break
        print("[", end="")
        for i in range(len(a)):
            print(command[i], end=", ")
        print("],")
    print()


[4, -1, -1, -1, -1, -1, -1, ],
[0, 223, 128, -1, -1, -1, -1, ],
[0, 223, 191, -1, -1, -1, -1, ],
[0, 176, 223, -1, -1, -1, -1, ],
[0, 128, 191, -1, -1, -1, -1, ],
[0, 128, 128, -1, -1, -1, -1, ],
[4, -1, -1, -1, -1, -1, -1, ],
[0, 206, 129, -1, -1, -1, -1, ],
[0, 206, 192, -1, -1, -1, -1, ],
[0, 145, 192, -1, -1, -1, -1, ],
[0, 145, 129, -1, -1, -1, -1, ],
[5, -1, -1, -1, -1, -1, 192, ],

[4, -1, -1, -1, -1, -1, -1, ],
[0, 223, 128, -1, -1, -1, -1, ],
[0, 223, 181, -1, -1, -1, -1, ],
[0, 128, 181, -1, -1, -1, -1, ],
[0, 128, 128, -1, -1, -1, -1, ],
[5, -1, -1, -1, -1, -1, 192, ],

[4, -1, -1, -1, -1, -1, -1, ],
[0, 223, 128, -1, -1, -1, -1, ],
[0, 223, 181, -1, -1, -1, -1, ],
[0, 128, 181, -1, -1, -1, -1, ],
[0, 128, 128, -1, -1, -1, -1, ],
[5, -1, -1, -1, -1, -1, 192, ],

[4, -1, -1, -1, -1, -1, -1, ],
[0, 223, 128, -1, -1, -1, -1, ],
[0, 223, 181, -1, -1, -1, -1, ],
[0, 128, 181, -1, -1, -1, -1, ],
[0, 128, 128, -1, -1, -1, -1, ],
[5, -1, -1, -1, -1, -1, 192, ],



In [39]:

seq2CAD(extrusion_splits_seq[5], name="5_test")

IndexError: list index out of range

In [40]:
for i, ext_seq in enumerate(extrusion_splits_seq):
    seq2CAD(ext_seq, name=str(i) + "_test")
    seq2pc(ext_seq, name=str(i) + "_test")
    


*******************************************************************
******        Statistics on Transfer (Write)                 ******

*******************************************************************
******        Transfer Mode = 0  I.E.  As Is       ******
******        Transferring Shape, ShapeType = 2                      ******
** WorkSession : Sending all data
 Step File Name : examples/0_test.step(774 ents)  Write  Done
Wrote stl file to examples/0_test.stl

*******************************************************************
******        Statistics on Transfer (Write)                 ******

*******************************************************************
******        Transfer Mode = 0  I.E.  As Is       ******
******        Transferring Shape, ShapeType = 2                      ******
Wrote stl file to examples/1_test.stl
** WorkSession : Sending all data
 Step File Name : examples/1_test.step(380 ents)  Write  Done

***************************************************

## Boolean

We can segment point clouds by their extrusions by iteratively building the model extrusion per extrusion and saving the labels. However there are problems with cut extrusions and interior walls in the final point cloud.

In [41]:
def combine_extrusions(base, extrusion):
    """Appends extrusion to base. Both are (60,17) from split_and_pad_sequence_by_extrusion()."""
    start = np.where(base == 3)[0][0]
    end = np.where(extrusion == 3)[0][0]
    combined = base
    combined[start:start+end, :] = extrusion[0:end, :]
    return combined

In [42]:
def extrusion_up_to(sequence, end_ext_idx):
    """Returns sequence up to the specified extrusion (inclusive). Extrusions are zero indexed."""
    extrusion = split_and_pad_sequence_by_extrusion(sequence)
    base = extrusion[0]
    for i in range(end_ext_idx):
        combined = combine_extrusions(base, extrusion[i+1])
        base = combined

    return base

In [43]:
seq = [
    [4, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
    [0, 223, 128, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
    [0, 223, 223, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
    [0, 128, 223, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
    [0, 128, 128, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
    [5, -1, -1, -1, -1, -1, 128, 128, 128, 128, 128, 0, 128, 256, 128, 0, 0], # This creates a 1x1x1 cube at the bottom-back-right position of the unit cube
    [4, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
    [0, 223, 128, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
    [0, 223, 223, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
    [0, 128, 223, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
    [0, 128, 128, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
    [5, -1, -1, -1, -1, -1, 128, 128, 128, 0, 128, 0, 128, 256, 128, 1, 0], # This creates a 1x1x1 cube at the bottom-back-right position of the unit cube
    [4, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
    [2, 128, 128, -1, -1, 95, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
    [5, -1, -1, -1, -1, -1, 128, 128, 128, 192, 192, 128, 64, 192, 128, 0, 0] # This adds a 0.5 high cylinder of radius 0.5 on top of the cube
]

In [44]:
# Cube with cylinder on top 
seq = [
    [4, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
    [0, 223, 128, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
    [0, 223, 223, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
    [0, 128, 223, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
    [0, 128, 128, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
    [5, -1, -1, -1, -1, -1, 128, 128, 128, 128, 128, 0, 128, 256, 128, 0, 0], # This creates a 1x1x1 cube at the bottom-back-right position of the unit cube
    [4, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
    [2, 128, 128, -1, -1, 95, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
    [5, -1, -1, -1, -1, -1, 128, 128, 128, 192, 192, 128, 64, 64, 128, 2, 0] # This adds a 0.5 high cylinder of radius 0.5 on top of the cube
]

In [45]:
import mesh_to_sdf
import trimesh

def remove_inner_points_with_sdf(points, mesh_path, threshold=0.0):
    """
    Removes all points that are inside the mesh by checking signed distance.
    :param points: Nx3 numpy array
    :param mesh_path: path to final STL mesh
    :param threshold: values < threshold are considered 'inside'
    :return: filtered_points, indices
    """
    mesh = trimesh.load(mesh_path)

    # Ensure the mesh is watertight
    if not mesh.is_watertight:
        print("Warning: Mesh is not watertight. SDF results may be unreliable.")

    sdf_values = mesh_to_sdf.mesh_to_sdf(mesh, points, surface_point_method='sample', sign_method='normal')

    keep_mask = sdf_values >= threshold  # only keep points outside or on surface
    return points[keep_mask], np.where(keep_mask)[0]


In [46]:
from OCC.Core.BRepAlgoAPI import BRepAlgoAPI_Cut
def segment_pc_into_extrusion(seq, nr_point=8096):
    extrusions = split_and_pad_sequence_by_extrusion(seq)
    all_points = []
    all_labels = []

    for i in range(len(extrusions)):

        if i > 0:
            print("\n prev")
            for a in extrusion_up_to(seq, i-1):
                print(a[0], end='')
            shape_prev = seq2shape(extrusion_up_to(seq, i-1))
        else:
            shape_prev = None

        print("\n curr")
    #    for a in extrusion_up_to(seq, i):
     #       print(a[0], end='')

        shape_curr = seq2shape(extrusion_up_to(seq, i))
        
    
        if shape_prev is not None:
            shape_diff = BRepAlgoAPI_Cut(shape_curr, shape_prev).Shape()
        else:
            shape_diff = shape_curr
       # if shape_prev is not None:
        #    shape2cad(shape_prev, f"shape_prev{i}")
   #     shape2cad(shape_curr, f"shape_curr{i}")
    #    shape2cad(shape_diff, f"shape_diff{i}")
        pc = CADsolid2pc(shape_diff, nr_point)
        labels = np.full((pc.shape[0],), i, dtype=int)
    
        all_points.append(pc)
        all_labels.append(labels)
        
    all_points = np.concatenate(all_points, axis=0)
    all_labels = np.concatenate(all_labels, axis=0)

    return all_points, all_labels

In [47]:
dataset = PointCloudEmbeddingSequenceDataset("../data", 'train', use_normals=False)

In [48]:
Not correct: 0, 3
Failed: 1, 2, 4

SyntaxError: invalid syntax (484727421.py, line 1)

In [49]:
data = get_data(dataset, 10)
for a in data[1]:
    print(a[0], end='   ')
print("")
for i, a in enumerate(data[1]):
    print(a[15], end=' ')
    if a[15] == 2:
        data[1][i][15] = 0
seq2CAD(data[1], "dataset")

4   0   0   0   0   0   0   5   4   0   0   0   0   5   4   0   0   0   0   5   4   0   0   0   0   5   4   0   0   0   0   5   3   3   3   3   3   3   3   3   3   3   3   3   3   3   3   3   3   3   3   3   3   3   3   3   3   3   3   3   
-1 -1 -1 -1 -1 -1 -1 0 -1 -1 -1 -1 -1 2 -1 -1 -1 -1 -1 2 -1 -1 -1 -1 -1 2 -1 -1 -1 -1 -1 2 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 
*******************************************************************
******        Statistics on Transfer (Write)                 ******

*******************************************************************
******        Transfer Mode = 0  I.E.  As Is       ******
******        Transferring Shape, ShapeType = 0                      ******
Wrote stl file to examples/dataset.stl
** WorkSession : Sending all data
 Step File Name : examples/dataset.step(960 ents)  Write  Done


In [50]:
for a in data[1]:
    print(a[15], end=' ')

-1 -1 -1 -1 -1 -1 -1 0 -1 -1 -1 -1 -1 0 -1 -1 -1 -1 -1 0 -1 -1 -1 -1 -1 0 -1 -1 -1 -1 -1 0 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 

In [51]:
#all_points, all_labels = segment_pc_into_extrusion(pad_seq(seq), 8096)
final_shape = seq2shape(pad_seq(seq))
shape2cad(final_shape, "final")
print(type(all_points), all_points.shape)
surface_points, surface_idx = remove_inner_points_with_sdf(all_points, "examples/final.stl")
surface_labels = all_labels[surface_idx]


*******************************************************************
******        Statistics on Transfer (Write)                 ******

*******************************************************************
******        Transfer Mode = 0  I.E.  As Is       ******
******        Transferring Shape, ShapeType = 0                      ******
** WorkSession : Sending all data
 Step File Name : examples/final.step(814 ents)  Write  Done
Wrote stl file to examples/final.stl


NameError: name 'all_points' is not defined

In [32]:
import open3d as o3d

def visualize_labeled_pc(points, labels):
    max_label = labels.max() + 1
    colors = plt.cm.get_cmap("tab10")(labels / max_label)[:, :3]

    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(points)
    pcd.colors = o3d.utility.Vector3dVector(colors)
    o3d.visualization.draw_geometries([pcd])


In [34]:
visualize_labeled_pc(surface_points, surface_labels)

/var/folders/97/07fn6f7n48j71zk65dskp3gm0000gn/T/ipykernel_7608/263153949.py:5: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  colors = plt.cm.get_cmap("tab10")(labels / max_label)[:, :3]


[Open3D WARNING] GLFW Error: Cocoa: Failed to find service port for display
[Open3D WARNING] GLFW Error: Cocoa: Failed to find service port for display


In [51]:
surface_points.shape

(18531, 3)

In [52]:
surface_labels.shape

(18531,)

## New idea

In [17]:
seq = [
    [4, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
    [0, 223, 128, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
    [0, 223, 223, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
    [0, 128, 223, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
    [0, 128, 128, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
    [5, -1, -1, -1, -1, -1, 128, 128, 128, 128, 128, 0, 128, 256, 128, 0, 0], # This creates a 1x1x1 cube at the bottom-back-right position of the unit cube
    [4, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
    [0, 223, 128, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
    [0, 223, 223, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
    [0, 128, 223, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
    [0, 128, 128, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
    [5, -1, -1, -1, -1, -1, 128, 128, 128, 0, 128, 0, 128, 256, 128, 1, 0], # This creates a 1x1x1 cube at the bottom-back-right position of the unit cube
    [4, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
    [2, 128, 128, -1, -1, 95, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
    [5, -1, -1, -1, -1, -1, 128, 128, 128, 192, 192, 128, 64, 64, 128, 2, 0] # This adds a 0.5 high cylinder of radius 0.5 on top of the cube
]
sequence = pad_seq(seq)

In [18]:
extrusion_splits_seq = split_and_pad_sequence_by_extrusion(sequence)

In [143]:
complete_pc = seq2pc(sequence, name=str(i) + "complete")
seq2CAD(sequence, "full")


*******************************************************************
******        Statistics on Transfer (Write)                 ******

*******************************************************************
******        Transfer Mode = 0  I.E.  As Is       ******
******        Transferring Shape, ShapeType = 0                      ******
Wrote stl file to examples/full.stl
** WorkSession : Sending all data
 Step File Name : examples/full.step(977 ents)  Write  Done


In [19]:
point_clouds = []
for i, ext_seq in enumerate(extrusion_splits_seq):
    pc = seq2pc(ext_seq, name=str(i) + "_test")
    point_clouds.append(pc)
merged_pc = np.concatenate(point_clouds, axis=0)
    
labels = []
for i, pc in enumerate(point_clouds):
    labels.append(np.full((pc.shape[0],), i, dtype=int))
merged_labels = np.concatenate(labels, axis=0)
  

In [52]:
from scipy.spatial import cKDTree

def filter_by_nearest_neighbor(pc_A, pc_B, epsilon=0.5):
    """
    Filters point cloud B using nearest neighbors from point cloud A.
    Keeps only points in B that are within `epsilon` distance to any point in A.
    
    :param pc_A: Nx3 point cloud (final model, no labels)
    :param pc_B: Mx3 point cloud (labeled extrusions)
    :return: filtered_pc_B, mask (M,), where mask[i] = True if point i is kept
    """
    tree = cKDTree(pc_A)
    distances, _ = tree.query(pc_B, k=1)

    mask = distances < epsilon
    return pc_B[mask], mask


In [53]:
# Assuming:
# pc_A: final model point cloud (Nx3)
# pc_B: merged extrusion point cloud (Mx3)
# labels_B: labels for each point in pc_B (Mx1)

filtered_pc, mask = filter_by_nearest_neighbor(complete_pc, merged_pc, epsilon=0.01)
filtered_labels = merged_labels[mask]

print("Remaining points:", filtered_pc.shape[0])


NameError: name 'complete_pc' is not defined

In [137]:
visualize_labeled_pc(filtered_pc, filtered_labels)

[Open3D WARNING] GLFW Error: Cocoa: Failed to find service port for display
[Open3D WARNING] GLFW Error: Cocoa: Failed to find service port for display


In [54]:
import open3d as o3d
import matplotlib.pyplot as plt

def visualize_labeled_pc(points, labels):
    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(points)

    colors = plt.cm.tab10(labels / labels.max())[:, :3]  # normalize
    pcd.colors = o3d.utility.Vector3dVector(colors)

    o3d.visualization.draw_geometries([pcd])

visualize_labeled_pc(merged_pc, merged_labels)


NameError: name 'merged_pc' is not defined

In [55]:
def print_sequence(sequence):
    """Input: (NxM) sequence matrix"""
    for a in sequence:
        print(a[0], end='')
        if a[0] == 3:
            break
        if a[0] == 5:
            print(" ", end="")
    print("")
    

In [56]:
def get_labled_pc_per_ext(sequence, nr_points=8096):
    """Input: sequence (60x17), Output: As many point clouds as there are extrusions, merged, each with a label"""
    extrusion_splits_seq = split_and_pad_sequence_by_extrusion(sequence)
    point_clouds = []
    for i, ext_seq in enumerate(extrusion_splits_seq):
        pc = seq2pc(ext_seq, name=str(i) + "_test", nr_points=nr_points)
        point_clouds.append(pc)
    merged_pc = np.concatenate(point_clouds, axis=0)
        
    labels = []
    for i, pc in enumerate(point_clouds):
        labels.append(np.full((pc.shape[0],), i, dtype=int))
    merged_labels = np.concatenate(labels, axis=0)
    return merged_pc, merged_labels

In [57]:
def save_pc(pc, path):
    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(pc)
    o3d.io.write_point_cloud(path, pcd)

In [58]:
from plyfile import PlyData, PlyElement
import numpy as np

def export_xyz_label_ply(points, labels, fname="examples/labeled_points.ply"):
    """
    Write a PLY with an extra uchar 'label' property.
    """
    points = np.asarray(points, dtype=np.float32)
    labels = np.asarray(labels, dtype=np.uint32)          # uint8/16/32 all fine

    ply_vertices = np.empty(len(points), 
                            dtype=[("x", "f4"), ("y", "f4"), ("z", "f4"), 
                                   ("label", "u4")])
    ply_vertices["x"] = points[:, 0]
    ply_vertices["y"] = points[:, 1]
    ply_vertices["z"] = points[:, 2]
    ply_vertices["label"] = labels

    el = PlyElement.describe(ply_vertices, "vertex")
    PlyData([el], text=True).write(fname)
    print(f"Saved {fname}")


### START

In [59]:
dataset = PointCloudEmbeddingSequenceDataset("../data", 'train', use_normals=False)

In [117]:
#data = get_data(dataset, 14)
#print(data[1].shape)
#print_sequence(data[1])

In [137]:
# Cube with cylinder on top 
seq_cut = [
    # This creates a 1x1x1 cube at the bottom-back-right position of the unit cube
    [4, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
    [0, 223, 128, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
    [0, 223, 223, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
    [0, 128, 223, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
    [0, 128, 128, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
    [5, -1, -1, -1, -1, -1, 128, 128, 128, 128, 128, 0, 128, 256, 128, 0, 0], 
    # This cuts a 0.5 high cylinder of radius 0.5 into the top of the cube
    [4, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
    [2, 128, 128, -1, -1, 95, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
    [5, -1, -1, -1, -1, -1, 128, 128, 128, 192, 192, 128, 64, 64, 128, 2, 0] 
]
seq_new_body = [
    [4, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
    [0, 223, 128, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
    [0, 223, 223, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
    [0, 128, 223, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
    [0, 128, 128, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
    [5, -1, -1, -1, -1, -1, 128, 128, 128, 128, 128, 0, 128, 256, 128, 0, 0], # This creates a 1x1x1 cube at the bottom-back-right position of the unit cube
    [4, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
    [2, 128, 128, -1, -1, 95, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
    [5, -1, -1, -1, -1, -1, 128, 128, 128, 192, 192, 128, 64, 196, 128, 0, 0] # This adds a 0.5 high cylinder of radius 0.5 on top of the cube
]
sequence = pad_seq(seq_new_body)

In [138]:
seq2CAD(sequence, "gt_cad") # data[1


*******************************************************************
******        Statistics on Transfer (Write)                 ******
Wrote stl file to examples/gt_cad.stl

*******************************************************************
******        Transfer Mode = 0  I.E.  As Is       ******
******        Transferring Shape, ShapeType = 0                      ******
** WorkSession : Sending all data
 Step File Name : examples/gt_cad.step(655 ents)  Write  Done


In [139]:
complete_pc = seq2pc(sequence, name="gt_pc")
merged_pc, merged_labels, extrusions = get_labled_pc_per_ext(sequence)

In [140]:
for i,s in enumerate(extrusions):
    seq2CAD(s, "test" + str(i))
    seq2pc(s, 10000, "test" + str(i))

Wrote stl file to examples/test0.stl
*******************************************************************
******        Statistics on Transfer (Write)                 ******

*******************************************************************
******        Transfer Mode = 0  I.E.  As Is       ******
******        Transferring Shape, ShapeType = 2                      ******
** WorkSession : Sending all data
 Step File Name : examples/test0.step(380 ents)  Write  Done


*******************************************************************
******        Statistics on Transfer (Write)                 ******

*******************************************************************
******        Transfer Mode = 0  I.E.  As Is       ******
Wrote stl file to examples/test1.stl
******        Transferring Shape, ShapeType = 2                      ******
** WorkSession : Sending all data
 Step File Name : examples/test1.step(148 ents)  Write  Done


In [106]:
# visualize_labeled_pc(merged_pc, merged_labels)

In [141]:
export_xyz_label_ply(merged_pc, merged_labels)

Saved examples/labeled_points.ply


In [142]:
filtered_pc, mask = filter_by_nearest_neighbor(complete_pc, merged_pc, epsilon=0.01)
filtered_labels = merged_labels[mask]

In [143]:
save_pc(filtered_pc, "examples/filtered_pc.ply")

In [144]:
export_xyz_label_ply(filtered_pc, filtered_labels, fname="examples/final_pc.ply")

Saved examples/final_pc.ply


In [79]:
visualize_labeled_pc(filtered_pc, filtered_labels)

/var/folders/97/07fn6f7n48j71zk65dskp3gm0000gn/T/ipykernel_9612/263153949.py:5: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  colors = plt.cm.get_cmap("tab10")(labels / max_label)[:, :3]


[Open3D WARNING] GLFW Error: Cocoa: Failed to find service port for display
[Open3D WARNING] GLFW Error: Cocoa: Failed to find service port for display


In [88]:
lol = []
for i in range (0, 1):
    try:
        num_ext = 0
        data = get_data(dataset, i)
        for command in data[1]:
            if command[0] == 5:
                num_ext += 1
        complete_pc = seq2pc(data[1], name="gt_pc", nr_points=num_ext*20000)
        merged_pc, merged_labels = get_labled_pc_per_ext(data[1], nr_points=20000)
        filtered_pc, mask = filter_by_nearest_neighbor(complete_pc, merged_pc, epsilon=0.005)
        filtered_labels = merged_labels[mask]
        visualize_labeled_pc(filtered_pc, filtered_labels)
    except Exception as e:
        print(e)
        lol.append(i)

/var/folders/97/07fn6f7n48j71zk65dskp3gm0000gn/T/ipykernel_9612/263153949.py:5: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  colors = plt.cm.get_cmap("tab10")(labels / max_label)[:, :3]


[Open3D WARNING] GLFW Error: Cocoa: Failed to find service port for display
[Open3D WARNING] GLFW Error: Cocoa: Failed to find service port for display


In [87]:
lol

[13, 44]

## Data Pipeline

MWE for cube with cylinder on top.

In [60]:
import os
import shutil
import h5py
import numpy as np
import sys
import pandas as pd
import json
import matplotlib.pyplot as plt
%matplotlib inline

sys.path.append("..")
sys.path.append("../code")

from dataset import PointCloudEmbeddingSequenceDataset
from models.DeepCAD.cadlib.visualize import vec2CADsolid
from OCC.Core.BRepCheck import BRepCheck_Analyzer
from OCC.Extend.DataExchange import write_step_file
from OCC.Core.STEPControl import STEPControl_Reader
from OCC.Core.StlAPI import StlAPI_Writer
from OCC.Core.BRepMesh import BRepMesh_IncrementalMesh
from models.DeepCAD.cadlib.extrude import CADSequence
from models.DeepCAD.cadlib.visualize import create_CAD
from models.DeepCAD.cadlib.visualize import CADsolid2pc
from models.DeepCAD.utils.pc_utils import write_ply
import open3d as o3d

In [61]:
def pad_seq(seq):
    """Takes custom sequence (N,17) and pads it to (60,17)"""
    eos_row = [3] + 16 * [-1]
    seq_np = np.array(seq, dtype=np.float32)
    num_pad_rows = 60 - seq_np.shape[0]
    pad_array = np.tile(eos_row, (num_pad_rows, 1))
    seq_np_pad = np.concatenate([seq_np, pad_array], axis=0)  
    return seq_np_pad

In [62]:
def seq2shape(seq):
    cad_seq = CADSequence.from_vector(seq, is_numerical=True)
    shape = create_CAD(cad_seq)
    return shape

In [63]:
def seq2pc(sequence, nr_points=8096):
    """ Input:  sequence:  Sequence as np (60,17)
                nr_points: Number of points to sample for the sequence as int
        Output: out_pc:    Output point cloud as np (N, 3)
    """
    
    shape = seq2shape(sequence)
    out_pc = CADsolid2pc(shape, n_points=nr_points)
    return out_pc

In [64]:
def get_gt_pc(sequence, n_points=20000):
    """ Input:  sequence: Sequence as np (60,17)
                n_points: Number of points which are sampled per extrusion in the sequence as int
        Output: Point cloud of the sequence as np (N, 3)
    """
    # Determine number of extrusions in sequence
    num_ext = 0
    for command in sequence:
        if command[0] == 5:
            num_ext += 1
    pc = seq2pc(sequence, nr_points=num_ext*n_points)
    return pc

In [65]:
def split_and_pad_sequence_by_extrusion(matrix, delimiter=5):
    """Takes (60,17) sequence and splits it by the extrusions and pads it and returns a (60,17) for each extrusion""" 
    matrix = np.array(matrix)
    assert matrix.shape == (60, 17), "Input must be (60, 17)"

    commands = matrix[:,0]
    split_indices = []
    start_idx = 0

    for idx, val in enumerate(commands):
        if val == delimiter:
            split_indices.append((start_idx, idx))
            start_idx = idx + 1

    output = []
    for start, end in split_indices:
        length = 59 - (end - start)
        pad_row = [[3] + 16 * [-1]] * length
        new_matrix = matrix[start:end+1]
       
        pad_matrix = np.vstack([new_matrix, pad_row])
        output.append(pad_matrix)
    return output

In [66]:
def get_labled_pc_per_ext(sequence, nr_points=8096):
    """ Input:  sequence: Sequence as np (60,17)
                nr_points: Number of points which are sampled per extrusion as int
        Output: merged_pc:            Point cloud where each point is labled by the extrusion that created it as np (N, 3)
                merged_labels:        Labels for each point as np (N)
                extrusion_splits_seq: Extrusion sequence per label as list of np (60,17)
    """
    
    extrusion_splits_seq = split_and_pad_sequence_by_extrusion(sequence)
    point_clouds = []
    for i, ext_seq in enumerate(extrusion_splits_seq):
        pc = seq2pc(ext_seq, nr_points=nr_points)
        point_clouds.append(pc)
    merged_pc = np.concatenate(point_clouds, axis=0)
        
    labels = []
    for i, pc in enumerate(point_clouds):
        labels.append(np.full((pc.shape[0],), i, dtype=int))
    merged_labels = np.concatenate(labels, axis=0)
    
    return merged_pc, merged_labels, extrusion_splits_seq

In [67]:
from scipy.spatial import cKDTree

def filter_by_nearest_neighbor(pc_A, pc_B, epsilon=0.5):
    """
    Filters point cloud B using nearest neighbors from point cloud A.
    Keeps only points in B that are within `epsilon` distance to any point in A.
    
    :param pc_A: Nx3 point cloud (gt_pc)
    :param pc_B: Mx3 point cloud (pc with labled extrusions)
    
    :return: pc_B filtered point cloud, mask (M,), where mask[i] = True if point i is kept
    """
    tree = cKDTree(pc_A)
    distances, _ = tree.query(pc_B, k=1)
    mask = distances < epsilon
    return pc_B[mask], mask


In [68]:
def filter_pc_and_labels(gt_pc, ext_pc, ext_labels, epsilon=0.01):
    """
    Input: gt_pc: Point cloud sampled from the original shape as np (N, 3)
           ext_pc: Point cloud where points are sampled from all extrusions as np (M, 3)
           ext_labels: Extrusion label for each point in ext_pc
    Output: filtered_pc: ext_pc where only points are retained that are within a distance of epsilon to a point in gt_pc
            filtered_labels: labels for each point in filtered_pc
    """
    filtered_pc, mask = filter_by_nearest_neighbor(gt_pc, ext_pc, epsilon=epsilon)
    filtered_labels = ext_labels[mask]
    return filtered_pc, filtered_labels

In [69]:
def save_pc(pc, path):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(pc)
    o3d.io.write_point_cloud(path, pcd)

In [70]:
def save_labels_with_ext(labels, ext_list, path):
    os.makedirs(os.path.dirname(h5_path), exist_ok=True)
    with h5py.File(h5_path, 'w') as f:
        f.create_dataset("labels", data=labels, compression='gzip')
    
        grp = f.create_group("sequences")
        for i, extr in enumerate(ext_list):
            grp.create_dataset(str(i), data=extr, compression='gzip')

In [71]:
def get_extrusion_distance_per_extrusion(data):
    distances_per_extrusion = []
    for item in data["sequence"]:
        if item["type"] == "ExtrudeFeature":
            
            extrude_id = item["entity"]
            extrude_entity = data["entities"][extrude_id]
            num_profiles = len(extrude_entity["profiles"])
            
            extent_one = extrude_entity["extent_one"]["distance"]["value"]
            extent_two = extrude_entity["extent_two"]["distance"]["value"]

            extrusion_distances = (extent_one, extent_two)

            for i in range(num_profiles):
                distances_per_extrusion.append(extrusion_distances)
    return distances_per_extrusion
    

In [72]:
def visualize_labeled_pc(points, labels):
    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(points)
    colors = plt.cm.tab10(labels / labels.max())[:, :3]
    pcd.colors = o3d.utility.Vector3dVector(colors)
    o3d.visualization.draw_geometries([pcd])

In [73]:
def load_and_visualize_target(id):
    """
    id: 8 digit id of the model as string
    """
    pc_path = os.path.join(DATA_DIR, "pc_from_vec", id[:4], id + ".ply")
    h5_path = os.path.join(DATA_DIR, "pc_from_vec_labels", id[:4], id + ".h5")

    pcd = o3d.io.read_point_cloud(pc_path)
    pc = np.asarray(pcd.points)

    with h5py.File(h5_path, 'r') as f:
        labels = f["labels"][:]
        sequences = {int(k): f["sequences"][k][:] for k in f["sequences"].keys()}

    num_classes = labels.max() + 1
    cmap = plt.cm.get_cmap("tab10", num_classes)

    for label_id, ext_seq in sequences.items():
        color_rgb = (np.array(cmap(label_id))[:3] * 255).astype(int)
        color_txt = rgb_colored_text(f"Extrusion {label_id}", color_rgb)
    
        print(f"{color_txt}")
        print("Sequence: ", end="")
        for command in ext_seq:
            if command[0] == 3:
                break
            print(int(command[0]), end="")
            if command[0]==5:
                if command[15] == 0:
                    print(" New")
                if command[15] == 1:
                    print(" Join")
                if command[15] == 2:
                    print(" Cut")
                if command[15] == 0:
                    print(" Intersect")
        print()

In [74]:
def rgb_colored_text(text, rgb):
    r, g, b = rgb
    return f"\033[38;2;{r};{g};{b}m{text}\033[0m"

In [75]:
dataset_train = PointCloudEmbeddingSequenceDataset("../data", 'train', use_normals=False)
dataset_val = PointCloudEmbeddingSequenceDataset("../data", 'validation', use_normals=False)
dataset_test = PointCloudEmbeddingSequenceDataset("../data", 'test', use_normals=False)
datasets = [dataset_train, dataset_val, dataset_test]

In [76]:
dataset_train.get_id(0), dataset_val.get_id(0)

('00675619', '00732275')

In [77]:
# Data creation settings

DATA_DIR = "data_exp"
nr_points_gt = 40000
nr_points_ext = 40000
epsilon = 0.005

In [78]:
def check_sequence(sequence, id):
    """
    Checks if the extrude distances are not zero, otherwise error.
    """
    json_path = os.path.join("..", "data", "cad_json", id[:4], id + ".json")  ### REFACTOR
    flag = False

    with open(json_path, "r") as file:
        data = json.load(file)
    distances = get_extrusion_distance_per_extrusion(data)

    ext_counter = 0
    for i, command in enumerate(sequence):
        if command[0] == 5: # only extrusion
            if command[12] == 0: # scale is numericalized at 0
                sequence[i][12] += 1
                flag = True
            if (command[16] == 0 or command[16] == 1) and command[13] == 128: # one-sided or symmetric
                if distances[ext_counter][0] > 0:
                    sequence[i][13] += 1
                    flag = True
                else:
                    sequence[i][13] -= 1
                    flag = True
            if command[16] == 2: # two-sided
                if command[13] == 128:
                    if distances[ext_counter][0] > 0:
                        sequence[i][13] += 1
                        flag = True
                    else:
                        sequence[i][13] -= 1
                        flag = True
                if command[14] == 128:
                    if distances[ext_counter][1] > 0:
                        sequence[i][14] += 1
                        flag = True
                    else:
                        sequence[i][14] -= 1
                        flag = True
            ext_counter += 1

    return sequence, flag 

In [79]:
import numpy as np

def adjust_pointcloud_to_fixed_size(points, labels, target_n=10000):
    """
    Adjust a point cloud and its labels to a fixed number of points.

    - If len(points) > target_n: randomly downsample
    - If len(points) < target_n: randomly upsample with replacement

    :param points: (N, 3) np.ndarray
    :param labels: (N,) np.ndarray
    :param target_n: int, desired number of points
    :return: (target_n, 3) points, (target_n,) labels
    """
    n = points.shape[0]

    if n == target_n:
        return points, labels
    elif n > target_n:
        idx = np.random.choice(n, target_n, replace=False)
    else:
        idx_extra = np.random.choice(n, target_n - n, replace=True)
        idx = np.concatenate([np.arange(n), idx_extra])

    return points[idx], labels[idx]


    return points[idx], labels[idx]


In [80]:
idx = np.random.choice(10, 15, replace=True)
idx

array([3, 5, 7, 6, 1, 8, 3, 2, 2, 3, 5, 7, 2, 9, 9])

In [81]:
i = 58

In [96]:
sequence = data['tgt_vec'].numpy()
sequence.shape

(60, 17)

In [97]:
data = dataset_test[i]
sequence = data['tgt_vec'].numpy()
id = data["id"]
# sequence = check_sequence(sequence, id)

#print(f"\rProcessing {i + 1}/{length} | ID: {id}", end="")
#sys.stdout.flush()

#pc_path = os.path.join(DATA_DIR, "pc_from_vec", id[:4], id + ".ply")
#h5_path = os.path.join(DATA_DIR, "pc_from_vec_labels", id[:4], id + ".h5")

gt_pc = get_gt_pc(sequence, n_points=nr_points_gt)
save_pc(gt_pc, "examples/gt_pc.ply")
labled_ext_pc, ext_labels, extrusions = get_labled_pc_per_ext(sequence, nr_points=nr_points_ext)
save_pc(labled_ext_pc, "examples/labled_pc.ply")
pc, labels = filter_pc_and_labels(gt_pc, labled_ext_pc, ext_labels, epsilon=epsilon)
save_pc(pc, "examples/pc.ply")

In [98]:
load_and_visualize_target(id)

[Open3D WARNING] Read PLY failed: unable to open file: data_exp/pc_from_vec/0061/00619236.ply


RPly: Unable to open file


FileNotFoundError: [Errno 2] Unable to open file (unable to open file: name = 'data_exp/pc_from_vec_labels/0061/00619236.h5', errno = 2, error message = 'No such file or directory', flags = 0, o_flags = 0)

In [99]:
visualize_labeled_pc(pc, labels)

[Open3D WARNING] GLFW Error: Cocoa: Failed to find service port for display
[Open3D WARNING] GLFW Error: Cocoa: Failed to find service port for display


In [24]:
error_dict = {}
num_points_dict = {}

for dataset in datasets:
    length = len(dataset)
    
    for i in range(length): #:
        
        try:
            data = dataset_train[i]
            sequence = data['tgt_vec'].numpy()
            id = data['id']
            sequence = check_sequence(sequence, id)
    
            print(f"\rProcessing {i + 1}/{length} | ID: {id}", end="")
            sys.stdout.flush()
    
            pc_path = os.path.join(DATA_DIR, "pc_from_vec", id[:4], id + ".ply")
            h5_path = os.path.join(DATA_DIR, "pc_from_vec_labels", id[:4], id + ".h5")
            
            gt_pc = get_gt_pc(sequence, n_points=nr_points_gt)
            labled_ext_pc, ext_labels, extrusions = get_labled_pc_per_ext(sequence, nr_points=nr_points_ext)
            pc, labels = filter_pc_and_labels(gt_pc, labled_ext_pc, ext_labels, epsilon=epsilon)
            
            num_points_dict[str(i)] = pc.shape[0]
    
         #   save_pc(pc, pc_path)
          #  save_labels_with_ext(labels, extrusions, h5_path)
    
          #  visualize_labeled_pc(pc, labels)
        except Exception as e:
            error_dict[str(i)] = e
    print()

Processing 420/160982 | ID: 00520940Warning: 4 faces have been skipped due to null triangulation
Processing 459/160982 | ID: 00884239Warning: 2 faces have been skipped due to null triangulation
Processing 582/160982 | ID: 00841692Warning: 1 face has been skipped due to null triangulation
Processing 698/160982 | ID: 00079994Warning: 2 faces have been skipped due to null triangulation
Processing 703/160982 | ID: 00787874Warning: 2 faces have been skipped due to null triangulation
Processing 1166/160982 | ID: 00289740Warning: 2 faces have been skipped due to null triangulation
Processing 1496/160982 | ID: 00070768Warning: 2 faces have been skipped due to null triangulation
Processing 1660/160982 | ID: 00372748Warning: 2 faces have been skipped due to null triangulation
Processing 1674/160982 | ID: 00587780Warning: 2 faces have been skipped due to null triangulation
Processing 1765/160982 | ID: 00005116Warning: 2 faces have been skipped due to null triangulation
Processing 2013/160982 | ID

KeyboardInterrupt: 

In [25]:
for k,v in error_dict.items():
    print(k, v)

250 box check failed
1021 box check failed
1101 Wrong number or type of arguments for overloaded function 'new_BRepAlgoAPI_Cut'.
  Possible C/C++ prototypes are:
    BRepAlgoAPI_Cut::BRepAlgoAPI_Cut()
    BRepAlgoAPI_Cut::BRepAlgoAPI_Cut(BOPAlgo_PaveFiller const &)
    BRepAlgoAPI_Cut::BRepAlgoAPI_Cut(TopoDS_Shape const &,TopoDS_Shape const &)
    BRepAlgoAPI_Cut::BRepAlgoAPI_Cut(TopoDS_Shape const &,TopoDS_Shape const &,BOPAlgo_PaveFiller const &,Standard_Boolean const)

3550 StdFail_NotDoneBRep_API: command not done raised from method Wire of class BRepBuilderAPI_MakeWire
3621 box check failed


In [54]:
load_and_visualize_target(id)

Extrusion 0
Sequence: 40000425 New
 Intersect



/var/folders/97/07fn6f7n48j71zk65dskp3gm0000gn/T/ipykernel_32799/3204524490.py:16: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  cmap = plt.cm.get_cmap("tab10", num_classes)


In [26]:
min_key = min(num_points_dict, key=num_points_dict.get)
print(f"Lowest: {min_key} ({num_points_dict[min_key]})")

Lowest: 725 (4373)


In [28]:
nump = []
for k,v in num_points_dict.items():
    nump.append(v)

In [30]:
plt.hist(nump, bins=50, range=(0, 200000), edgecolor='black')
plt.title("Histogram of Values (0 to 50,000)")
plt.xlabel("Value")
plt.ylabel("Frequency")
plt.show()

In [101]:
lol = []
for i in range(0,180000):
    lol.append(i)

In [102]:
len(lol)

180000

In [20]:
def load_data(id):
    """
    id: 8 digit id of the model as string
    """
    pc_path = os.path.join(DATA_DIR, "pc_from_vec", id[:4], id + ".ply")
    h5_path = os.path.join(DATA_DIR, "pc_from_vec_labels", id[:4], id + ".h5")

    pcd = o3d.io.read_point_cloud(pc_path)
    pc = np.asarray(pcd.points)

    print(pc.shape)

    with h5py.File(h5_path, 'r') as f:
        labels = f["labels"][:]
        sequences = {int(k): f["sequences"][k][:] for k in f["sequences"].keys()}

    print(pc.shape, labels.shape)
    for k, v in sequences.items():
        print(k, v.shape)

    num_classes = labels.max() + 1
    cmap = plt.cm.get_cmap("tab10", num_classes)

    for label_id, ext_seq in sequences.items():
        color_rgb = (np.array(cmap(label_id))[:3] * 255).astype(int)
        color_txt = rgb_colored_text(f"Extrusion {label_id}", color_rgb)
    
        print(f"{color_txt}")
        print("Sequence: ", end="")
        for command in ext_seq:
            if command[0] == 3:
                break
            print(int(command[0]), end="")
            if command[0]==5:
                if command[15] == 0:
                    print(" New")
                if command[15] == 1:
                    print(" Join")
                if command[15] == 2:
                    print(" Cut")
                if command[15] == 0:
                    print(" Intersect")
        print()
    return pc, labels, sequences

In [63]:
for dataset in datasets:
    length = len(dataset)
    for i in range(length):
        id = dataset.get_id(i)
        pc, labels, seqs = load_data(id)
        visualize_labeled_pc(pc, labels)

(10000, 3)
(10000, 3) (10000,)
0 (60, 17)
1 (60, 17)
2 (60, 17)
3 (60, 17)
4 (60, 17)
Extrusion 0
Sequence: 400042424242425 New
 Intersect

Extrusion 1
Sequence: 425 Join

Extrusion 2
Sequence: 425 Join

Extrusion 3
Sequence: 425 Join

Extrusion 4
Sequence: 425 Join

[Open3D WARNING] GLFW Error: Cocoa: Failed to find service port for display
[Open3D WARNING] GLFW Error: Cocoa: Failed to find service port for display


/var/folders/97/07fn6f7n48j71zk65dskp3gm0000gn/T/ipykernel_25906/2360958999.py:22: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  cmap = plt.cm.get_cmap("tab10", num_classes)


(10000, 3)
(10000, 3) (10000,)
0 (60, 17)
1 (60, 17)
2 (60, 17)
3 (60, 17)
4 (60, 17)
Extrusion 0
Sequence: 401015 New
 Intersect

Extrusion 1
Sequence: 4015 Join

Extrusion 2
Sequence: 4105 Join

Extrusion 3
Sequence: 42425 Join

Extrusion 4
Sequence: 400005 Cut

[Open3D WARNING] GLFW Error: Cocoa: Failed to find service port for display
[Open3D WARNING] GLFW Error: Cocoa: Failed to find service port for display


KeyboardInterrupt: 

In [21]:
DATA_DIR = "../data"
pc, labels, seqs = load_data("00675619")

(10000, 3)
(10000, 3) (10000,)
0 (60, 17)
1 (60, 17)
2 (60, 17)
3 (60, 17)
4 (60, 17)
Extrusion 0
Sequence: 400042424242425 New
 Intersect

Extrusion 1
Sequence: 425 Join

Extrusion 2
Sequence: 425 Join

Extrusion 3
Sequence: 425 Join

Extrusion 4
Sequence: 425 Join



/var/folders/97/07fn6f7n48j71zk65dskp3gm0000gn/T/ipykernel_25906/2360958999.py:22: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  cmap = plt.cm.get_cmap("tab10", num_classes)


In [22]:
visualize_labeled_pc(pc, labels)

[Open3D WARNING] GLFW Error: Cocoa: Failed to find service port for display
[Open3D WARNING] GLFW Error: Cocoa: Failed to find service port for display


In [205]:
seqs[0].shape

(60, 17)

In [ ]:
with h5py.File(h5_path, 'r') as f:
    labels = f["labels"][:]
    sequences = {int(k): f["sequences"][k][:] for k in f["sequences"].keys()}

In [27]:
dataset_train = PointCloudEmbeddingSequenceDataset("../data", 'train', use_normals=False)
dataset_val = PointCloudEmbeddingSequenceDataset("../data", 'validation', use_normals=False)
dataset_test = PointCloudEmbeddingSequenceDataset("../data", 'test', use_normals=False)
datasets = [dataset_train, dataset_val, dataset_test]

In [56]:
from tqdm import tqdm  
import numpy as np
import h5py

counter = 0
eos_row = np.array([3] + 16 * [-1])  

for dataset in datasets:
    length = len(dataset)
    for i in tqdm(range(length)):
        h5_path = dataset.get_cad_seq_path(i)
        with h5py.File(h5_path, 'r+') as f:  # Open in read/write mode
            seq = f['vec'][:]
            if seq[-1][0] != 3:
                seq = np.concatenate((seq, eos_row[None, :]), axis=0)
                del f['vec']
                f.create_dataset('vec', data=seq, compression='gzip', dtype=seq.dtype)
                counter += 1


100%|█████████████████████████████████████| 8038/8038 [00:03<00:00, 2428.53it/s]


In [57]:
counter

3684

In [54]:
eos_row = [3] + 16 * [-1]
eos_row.shape

AttributeError: 'list' object has no attribute 'shape'

In [61]:
from tqdm import tqdm  
import numpy as np
import h5py

max_extrusion_counter = 0

for dataset in datasets:
    length = len(dataset)
    for i in tqdm(range(length)):
        extrusion_counter = 0
        h5_path = dataset.get_cad_seq_path(i)
        with h5py.File(h5_path, 'r') as f:  # Open in read/write mode
            seq = f['vec'][:]
        for command in seq:
            if command[0] == 5:
                extrusion_counter += 1
        if extrusion_counter > max_extrusion_counter:
            max_extrusion_counter = extrusion_counter

100%|█████████████████████████████████████| 8038/8038 [00:02<00:00, 3281.19it/s]


In [62]:
max_extrusion_counter

10

In [ ]:
counter = 0
for dataset in datasets:
    length = len(dataset)
    for i in range(length):
        h5_path = dataset.get_cad_seq_path(i)
        with h5py.File(h5_path, 'a') as f:
            if old_key in f:
                f.move(old_key, new_key)
                counter += 1
print(counter)

In [36]:
old_key = "sequence"
new_key = "vec"
counter = 0
for dataset in datasets:
    length = len(dataset)
    for i in range(length):
        h5_path = dataset.get_cad_seq_path(i)
        with h5py.File(h5_path, 'a') as f:
            if old_key in f:
                f.move(old_key, new_key)
                counter += 1
print(counter)

3684


In [2]:
import pickle
with open("../code/error_dict.pkl", "rb") as f:
    model = pickle.load(f)

print(len(model))

159


In [45]:
for k,v in model.items():
    print(k,v)


30 Updated sequence for ../data/cad_vec/0090/00901282.h5 with id 00901282.
6 Updated sequence for ../data/cad_vec/0005/00050520.h5 with id 00050520.
29 Updated sequence for ../data/cad_vec/0095/00952806.h5 with id 00952806.
36 Updated sequence for ../data/cad_vec/0067/00676653.h5 with id 00676653.
19 Updated sequence for ../data/cad_vec/0057/00571389.h5 with id 00571389.
11 Updated sequence for ../data/cad_vec/0022/00223430.h5 with id 00223430.
24 Updated sequence for ../data/cad_vec/0062/00620381.h5 with id 00620381.
10 Updated sequence for ../data/cad_vec/0049/00495851.h5 with id 00495851.
22 Updated sequence for ../data/cad_vec/0011/00116780.h5 with id 00116780.
8 Updated sequence for ../data/cad_vec/0001/00016936.h5 with id 00016936.
3 Updated sequence for ../data/cad_vec/0065/00651513.h5 with id 00651513.
49 Updated sequence for ../data/cad_vec/0041/00412922.h5 with id 00412922.
18 Updated sequence for ../data/cad_vec/0008/00086498.h5 with id 00086498.
26 Updated sequence for ../d